# 04 — Transformer Kernels & Profiling Intro

Run the **transformer kernel benchmark** (QKV, Softmax, LayerNorm, GELU, MLP) and a short **profiling** intro. RTX 3050–friendly (light config).

In [2]:
import sys
from pathlib import Path
root = Path("..").resolve()
sys.path.insert(0, str(root))
import torch

## Run transformer benchmark (light)

This calls the same logic as `python benchmarks/transformer_benchmark.py` with default (reduced) load.

In [3]:
if torch.cuda.is_available():
    # Import and run benchmark sections (light: B=8, S=256)
    from benchmarks.transformer_benchmark import (
        bench_qkv, bench_softmax, bench_layernorm, bench_gelu, bench_mlp, _rest
    )
    print("Transformer kernels (FP16, B=8 S=256 H=768)\n")
    bench_qkv()
    _rest()
    bench_softmax()
    _rest()
    bench_layernorm()
    _rest()
    bench_gelu()
    _rest()
    bench_mlp()
    print("\nDone.")
else:
    print("CUDA not available.")

Transformer kernels (FP16, B=8 S=256 H=768)


--- Fused QKV projection (FP16) ---
  PyTorch: latency=1.6817 ms, mem_bandwidth=9.59 GB/s, throughput=4309.7 GFLOPS
  Triton: latency=2.0905 ms, mem_bandwidth=7.71 GB/s, throughput=3467.1 GFLOPS
  CUDA: skipped (default low load; set TRANSFORMER_BENCHMARK_CUDA=1 to enable)

--- Softmax (last dim, FP16) ---
  PyTorch: latency=0.2000 ms, mem_bandwidth=31.46 GB/s
  Triton: latency=0.6628 ms, mem_bandwidth=9.49 GB/s
  CUDA: skipped (default low load; set TRANSFORMER_BENCHMARK_CUDA=1 to enable)

--- LayerNorm (FP16) ---
  PyTorch: latency=0.1973 ms, mem_bandwidth=31.90 GB/s
  Triton: latency=0.1987 ms, mem_bandwidth=31.68 GB/s
  CUDA: skipped (default low load; set TRANSFORMER_BENCHMARK_CUDA=1 to enable)

--- GELU (FP16) ---
  PyTorch: latency=0.2052 ms, mem_bandwidth=30.65 GB/s
  Triton: latency=0.2002 ms, mem_bandwidth=31.43 GB/s
  CUDA: skipped (default low load; set TRANSFORMER_BENCHMARK_CUDA=1 to enable)

--- Fused MLP block (FP16) ---
  Py

## Profiling (optional)

- **Nsight Systems**: timeline → `python profiling/run_nsight_profiling.py` (set `SKIP_NSYS=1` to skip nsys; `NCU_QUICK=1` for ~1 min ncu).
- **Roofline**: `python profiling/roofline_analysis/plot_roofline.py` → `profiling/nsight_reports/roofline_model.png`.

See `profiling/nsight_reports/README.md` and `docs/optimization_guide.md`.

In [ ]:
# Quick roofline from notebook (run from repo root)
try:
    from profiling.roofline_analysis.plot_roofline import plot_roofline
    p = plot_roofline(B=8, S=256, H=768)
    print("Roofline saved:", p)
except Exception as e:
    print("Run from repo root:", e)